In [ ]:
print("=" * 72)
print("STEP 0 - Clone repo, install Python dependencies")
print("=" * 72)

!git clone -q https://github.com/uzairlol/ELICIT-fyp.git /kaggle/working/elicit
%cd /kaggle/working/elicit

!git log -1 --format="Cloned HEAD  : %h %s"
!git status -sb | head -1

print("Environment pre-check:")
!python --version
!free -h | head -2
!df -h /kaggle | tail -1

!pip install -qq -r requirements.txt
print("requirements.txt installed.")

In [ ]:
print("=" * 72)
print("STEP 1 - Install vLLM and verify CUDA/GPU")
print("=" * 72)

!apt-get update -y -qq > /dev/null
!apt-get install -y -qq pciutils zstd > /dev/null
print("apt: pciutils + zstd installed.")

# vLLM (on a Kaggle GPU T4 x2 session). Requires an internet-enabled session.
!pip install -qq vllm
print("vLLM installed. Versions + CUDA check:")
!python -c "import vllm, torch; print('vllm =', vllm.__version__, '| torch =', torch.__version__, '| cuda_available =', torch.cuda.is_available(), '| gpus =', torch.cuda.device_count())"

!nvidia-smi

In [ ]:
# GGUF Q4_K_M = the same ~9 GB weights as `qwen2.5:14b` in Ollama (drop-in
# behavioural equivalent), served on one T4 with KV cache fully in VRAM.
#
# For the ~2x faster dual-GPU path (GPTQ int4, also ~9 GB) instead run:
#   python -m vllm.entrypoints.openai.api_server \
#     --model Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4 \
#     --served-model-name qwen2.5-14b --tensor-parallel-size 2 \
#     --max-model-len 8192 --max-num-seqs 32 --gpu-memory-utilization 0.90

import json
import subprocess
import time
import urllib.request

print("=" * 72)
print("STEP 2 - Start vLLM server in the background")
print("=" * 72)

cmd = [
    "python",
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    "Qwen/Qwen2.5-14B-Instruct-GGUF:Qwen2.5-14B-Instruct-Q4_K_M.gguf",
    "--served-model-name",
    "qwen2.5-14b",
    "--max-model-len",
    "8192",
    "--max-num-seqs",
    "16",
    "--gpu-memory-utilization",
    "0.92",
    "--swap-space",
    "0",
]

print("Launching vLLM with:")
print("  " + " ".join(cmd))
with open("/kaggle/working/vllm.log", "wb") as log_handle:
    vllm_proc = subprocess.Popen(
        cmd,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )
print(f"vLLM PID: {vllm_proc.pid}   log: /kaggle/working/vllm.log")

started_at = time.time()
time.sleep(20)  # let the engine start downloading weights before polling
print("Waiting for vLLM to serve /v1/models (download + load can take minutes) ...")
ready = False
for attempt in range(1, 301):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=3) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        pass
    if attempt % 30 == 0:
        elapsed = time.time() - started_at
        tail = subprocess.run(
            ["tail", "-3", "/kaggle/working/vllm.log"],
            capture_output=True,
            text=True,
        ).stdout.strip()
        print(f"  ...not ready after {elapsed:.0f}s ({attempt} polls); log tail:")
        if tail:
            print("  " + tail.replace("\n", "\n  "))
    time.sleep(2)

if ready:
    elapsed = time.time() - started_at
    print(f"vLLM is READY after {elapsed:.1f}s.")
    with urllib.request.urlopen("http://127.0.0.1:8000/v1/models") as response:
        models = json.loads(response.read())
    for model in models.get("data", []):
        print(f"  serving model: {model.get('id')}")
else:
    print("Timed out after 10 min; check /kaggle/working/vllm.log")

In [ ]:
import json
import urllib.request

print("=" * 72)
print("STEP 3 - GPU + model smoke test")
print("=" * 72)

!nvidia-smi
print("VRAM free (MiB), via torch:")
!python -c "import torch; print([round(torch.cuda.mem_get_info(i)[0] / 1024**2) for i in range(torch.cuda.device_count())])"

print("vLLM log tail:")
!tail -8 /kaggle/working/vllm.log

# End-to-end smoke test: one tiny chat completion through the served engine.
body = json.dumps(
    {
        "model": "qwen2.5-14b",
        "messages": [{"role": "user", "content": "Reply with exactly: READY"}],
        "max_tokens": 8,
        "temperature": 0.0,
    }
).encode("utf-8")
request = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=body,
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=60) as response:
    reply = json.loads(response.read())
print(f"Smoke test reply: {reply['choices'][0]['message']['content'].strip()!r}")

In [ ]:
print("=" * 72)
print("STEP 4 - Verify configuration, then run the simulation")
print("=" * 72)

import sys  # noqa: E402  (import must run after the appending sys.path below)

sys.path.append("/kaggle/working/elicit/src")
%cd /kaggle/working/elicit/src

# Resolve the config the simulation will actually use (must match the vLLM
# --served-model-name above).
from core import parameters  # noqa: E402  (after sys.path.append / %cd)
from llm import create_llm_client  # noqa: E402

client = create_llm_client()
print("Resolved runtime configuration:")
print(f"  LLM_BACKEND           = {parameters.LLM_BACKEND}")
print(f"  LLM_MODEL             = {parameters.LLM_MODEL}")
print(f"  LLM_BASE_URL          = {parameters.LLM_BASE_URL}")
print(f"  client class          = {type(client).__name__}")
print(f"  LLM_MAX_CONCURRENCY   = {parameters.LLM_MAX_CONCURRENCY}")
print(f"  TOM_MAX_CONCURRENCY   = {parameters.TOM_MAX_CONCURRENCY}")
print(f"  client max_inflight   = {client.max_concurrency}")
print(f"  SCENARIO              = {getattr(parameters, 'SCENARIO', '<unset>')}")
print(f"  LDF_ENABLED           = {getattr(parameters, 'LDF_ENABLED', False)}")
print(f"  CLIMATE_SHOCK_ENABLED = {getattr(parameters, 'CLIMATE_SHOCK_ENABLED', False)}")
print(f"  DEMOCRACY_INTERVAL    = {getattr(parameters, 'DEMOCRACY_INTERVAL', '<unset>')}")
print(f"  NUM_AGENTS            = {getattr(parameters, 'NUM_AGENTS', '<unset>')}")

# Run seed 2 (adjust seeds or rounds as needed). --model-name must match
# the --served-model-name used to start vLLM.
run_cmd = (
    "python run_experiments.py "
    "--scenario ldf --enable-ldf --enable-climate-shocks "
    "--model-name qwen2.5-14b --seeds 2 --num-rounds 30"
)
print("SIMULATION COMMAND:")
print(f"  {run_cmd}")
print("=" * 72)
print("RUN STARTING - first-round output below")
print("=" * 72)
!python run_experiments.py --scenario ldf --enable-ldf --enable-climate-shocks --model-name qwen2.5-14b --seeds 2 --num-rounds 30

In [ ]:
print("=" * 72)
print("STEP 5 - Archive results")
print("=" * 72)

!rm -rf /kaggle/working/simulation_results.tar.gz
!tar -czvf /kaggle/working/simulation_results.tar.gz /kaggle/working/elicit/results/ /kaggle/working/elicit/data/
print("Archive size:")
!du -h /kaggle/working/simulation_results.tar.gz
print("Result files:")
!ls -R /kaggle/working/elicit/results | head -40